In [ ]:
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm

import sys
sys.path.append('../src')
import helper

def compute_heatmap(dataset, run): #collects logits (bank predictions) and compares them with labels (bank ground_truth)
    bank = pd.read_pickle(f'../results/{dataset}/{run}/rejectors/bank.pkl')
    bank = bank.rename(columns=lambda c: c.lower().replace("__", "_"))
    bank = bank.drop(columns=['prompt', 'sample_id', 'eval_name'])
    
    folders = glob(f'../results/{dataset}/{run}/bank/*')
    targets = bank.keys()
    
    res=[]
    for folder in tqdm(folders):
        model_name = folder.split('/')[-1].split('.npy')[0]
        logits = np.load(f'{folder}')
        source = model_name.lower().replace('__', '_')
        for target in targets:
            labels = bank[target]
            
            mask = ~labels.isna().values
            m = helper.compute_eval_metrics(logits[mask], labels[mask].values.astype(int))
            res.append([source, target, m])
    
    rows = []
    for source, target, metrics in res:
        row = {'source': source, 'target': target}
        row.update(metrics)         
        rows.append(row)
    
    df = pd.DataFrame(rows)
    return df

In [ ]:
RUNS = 10
datasets = ['openLLMLeaderboard', 'routerBench']
for dataset in datasets:
    for run in range(RUNS):
        df = compute_heatmap(dataset, run)
        df[['source', 'target', 'auroc', 'brier', 'aupr']].to_pickle(f'../results/{dataset}/{run}/heatmap_compressed.pkl')